# 23N Field Tests #09 · Can small open models understand spoken Gujarati?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/ft09-gujarati-speech-small-model/ft09-gujarati-speech-small-model/ft09_gujarati_speech_small_model.ipynb)

**Question.** Voice is how many people would rather reach a phone line, a shop or a government helpdesk, and
Gujarati is the mother tongue of about 5.5 crore people (Census 2011). How well do small open speech-recognition
models, running on a free GPU, transcribe spoken Gujarati, and where do they fail?

**Method.** The notebook takes the *same sentences* read aloud in Gujarati, Hindi and English from Google's FLEURS
test set (FLEURS sentences are parallel translations, so the content is held constant across languages). It
transcribes up to 150 clips per language with four open models and scores each with word error rate (WER) and
character error rate (CER), using one script-safe text normaliser for every language. It then looks at *where* the
errors are: numbers, long sentences, runaway output and the most common word swaps.

| Model | Size | Languages | Licence |
|---|---|---|---|
| `openai/whisper-small` | ~0.24B | multilingual | MIT |
| `openai/whisper-large-v3-turbo` | ~0.81B | multilingual | MIT |
| `facebook/mms-1b-all` (language adapters) | ~1B | 1,162 languages | **CC-BY-NC 4.0 (non-commercial)** |
| `vasista22/whisper-gujarati-small` | ~0.24B | Gujarati fine-tune of Whisper small | Apache 2.0 |

**Data.** [FLEURS](https://huggingface.co/datasets/google/fleurs) (Google), test split, configs `gu_in`, `hi_in`,
`en_us`. Licence **CC BY 4.0**, no login needed. Audio is downloaded from Hugging Face at run time (about 0.5 GB
per language) and is not redistributed here. `vasista22/whisper-gujarati-small` was fine-tuned on FLEURS
*train + dev*, so the FLEURS *test* clips used here are held out for it, but it has seen the same speakers' style
of read speech: treat its number as a best case.

**Runtime.** Free Colab T4: about 15–20 minutes for Run all (downloads ~6 GB, ~1,050 transcriptions, video).
Without a GPU the notebook drops to whisper-small on 30 clips per language and flags the run as not postable.

*Independent study on public data by 23N AI Labs. Not affiliated with or endorsed by Google, OpenAI, Meta or any
model author mentioned here.*

## 1 · Setup

In [ ]:
# CELL: system_deps
# libraqm gives Pillow complex-script shaping, which Gujarati and Hindi need (matras and conjuncts).
!apt-get -qq install -y libraqm0 libfribidi0 > /dev/null 2>&1 || echo "apt not available, continuing"
%pip install -q jiwer soundfile gradio

In [ ]:
# CELL: config
import os, sys, io, re, json, math, time, textwrap, unicodedata, subprocess, warnings, tarfile, gc, collections, zlib
import urllib.request
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message="Glyph .* missing")

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Test switch. OFF by default. Only for build checks without internet/GPU: it replaces the FLEURS download with
# a few synthetic clips and the four models with deterministic stand-ins. Its output is NOT real. Never post it.
SMOKE_TEST = os.environ.get("FT09_SMOKE_TEST", "0") == "1"

# ---- Assumptions and knobs (all named, all printed with the result) ----
LANGS = {  # FLEURS config -> (label, Whisper language code, MMS adapter code, main script)
    "gu_in": ("Gujarati", "gu", "guj", "Gujarati"),
    "hi_in": ("Hindi", "hi", "hin", "Devanagari"),
    "en_us": ("English", "en", "eng", "Latin"),
}
MODELS = [  # key, Hugging Face id, family, licence, languages it is evaluated on
    ("whisper-small", "openai/whisper-small", "whisper", "MIT", ["gu_in", "hi_in", "en_us"]),
    ("whisper-large-v3-turbo", "openai/whisper-large-v3-turbo", "whisper", "MIT", ["gu_in", "hi_in", "en_us"]),
    ("mms-1b-all", "facebook/mms-1b-all", "mms", "CC-BY-NC-4.0 (non-commercial)", ["gu_in", "hi_in", "en_us"]),
    ("whisper-gujarati-small", "vasista22/whisper-gujarati-small", "whisper", "Apache-2.0", ["gu_in"]),
]
FINETUNED_GU = "whisper-gujarati-small"   # trained on FLEURS train+dev: a best case, flagged in every output
N_PER_LANG = 150          # clips per language (one recording per sentence, same sentences across languages if possible)
N_PER_LANG_CPU = 30       # used automatically without a GPU (and only whisper-small); such a run is not postable
N_MIN = 100               # fewer usable clips than this per language → the run is flagged
SEED = 23
MAX_AUDIO_S = 30.0        # Whisper reads 30 s windows; longer clips are trimmed (counted and printed)
BATCH = 16
MAX_NEW_TOKENS = 225      # Whisper decoding limit per clip
RUNAWAY_RATIO = 2.0       # an output with more than this × the reference word count (and 10+ extra words) = runaway
EN_WER_CEILING = 0.15     # assumption: on read English FLEURS speech a large-v3 model should be far below this
REPORTED_FT_WER = 0.1898  # WER on FLEURS gu_in test reported on the model card of vasista22/whisper-gujarati-base
FT_WER_TOLERANCE = 2.0    # our small-model WER above this × the reported base-model WER → our scoring disagrees with theirs
N_BOOT = 1000             # bootstrap resamples for confidence intervals

# ---- Where things live (this branch until it is merged) ----
REPO_REF = "ft09-gujarati-speech-small-model"
RAW = f"https://raw.githubusercontent.com/23nlabs/field-tests/{REPO_REF}"
FLEURS_REPO = "google/fleurs"
FT_NUM = "09"
DATA_CREDIT = "Data: Google FLEURS test set, CC BY 4.0"

# ---- Brand kit ----
if not os.path.exists("brand23n.py"):
    urllib.request.urlretrieve(f"{RAW}/brand/brand23n.py", "brand23n.py")
import brand23n as B
B.setup()
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    HARDWARE = torch.cuda.get_device_name(0) if HAS_GPU else "CPU"
except Exception:
    HAS_GPU, HARDWARE = False, "CPU"
if SMOKE_TEST:
    HARDWARE = "CPU (stand-in)"
if not HAS_GPU and not SMOKE_TEST:
    N_PER_LANG = N_PER_LANG_CPU
    MODELS = [m for m in MODELS if m[0] == "whisper-small"]
    print("!! No GPU: reduced run (whisper-small only, %d clips per language). Switch to a T4 for the headline." % N_PER_LANG)
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| GPU:", HAS_GPU, "| matplotlib", matplotlib.__version__, "| SMOKE_TEST:", SMOKE_TEST)
if SMOKE_TEST:
    print("!! SMOKE TEST MODE: synthetic clips and stand-in models. These results are NOT real. Do NOT post.")

In [ ]:
# CELL: fonts
# Gujarati and Hindi text. IBM Plex has no Indic glyphs, so those strings are drawn with Noto Sans Gujarati or
# Noto Sans Devanagari (SIL OFL, from the Google Fonts GitHub repo; both also cover Latin and digits). matplotlib
# doesn't shape complex scripts reliably, so they are rendered with Pillow + libraqm (HarfBuzz) and pasted in.
from PIL import Image, ImageDraw, ImageFont, features
os.makedirs("fonts", exist_ok=True)
FONT_URLS = {
    "Gujarati": ("fonts/NotoSansGujarati-VF.ttf",
                 "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansgujarati/NotoSansGujarati%5Bwdth%2Cwght%5D.ttf"),
    "Devanagari": ("fonts/NotoSansDevanagari-VF.ttf",
                   "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansdevanagari/NotoSansDevanagari%5Bwdth%2Cwght%5D.ttf"),
}
for path, url in FONT_URLS.values():
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
HAS_RAQM = features.check("raqm")
if not HAS_RAQM:
    print("WARNING: Pillow has no libraqm, so Gujarati will be drawn WITHOUT shaping.\n"
          "Fix: run `!apt-get install -y libraqm0 libfribidi0`, then Runtime → Restart session, then Run all.")

SCRIPT_RANGES = {"Gujarati": ("઀", "૿"), "Devanagari": ("ऀ", "ॿ")}
def script_share(s, script):
    """Share of letters in `s` that belong to `script` (Latin = a-z)."""
    letters = [c for c in str(s) if c.isalpha() or unicodedata.category(c) in ("Mn", "Mc")]
    if not letters:
        return 0.0
    if script == "Latin":
        return sum(c.isascii() for c in letters) / len(letters)
    lo, hi = SCRIPT_RANGES[script]
    return sum(lo <= c <= hi for c in letters) / len(letters)

_font_cache = {}
def _pil_font(px, bold=False, script="Gujarati"):
    key = (int(px), bold, script)
    if key not in _font_cache:
        layout = ImageFont.Layout.RAQM if HAS_RAQM else ImageFont.Layout.BASIC
        f = ImageFont.truetype(FONT_URLS[script][0], int(px), layout_engine=layout)
        try:
            f.set_variation_by_axes([600 if bold else 400, 100])   # axes: wght, wdth
        except Exception:
            pass
        _font_cache[key] = f
    return _font_cache[key]

def text_image(text, px, color=B.INK, bold=False, max_width=None, line_gap=1.3, max_lines=None):
    """Render text (any of our scripts) to an RGBA array with proper shaping, wrapped to max_width pixels."""
    script = "Devanagari" if script_share(text, "Devanagari") > script_share(text, "Gujarati") else "Gujarati"
    font = _pil_font(px, bold, script)
    lines = []
    for para in str(text).split("\n"):
        cur = ""
        for w in para.split(" "):
            trial = (cur + " " + w).strip()
            if max_width and cur and font.getlength(trial) > max_width:
                lines.append(cur); cur = w
            else:
                cur = trial
        lines.append(cur)
    if max_lines and len(lines) > max_lines:
        lines = lines[:max_lines]; lines[-1] = lines[-1].rstrip(" .,") + " …"
    lh = int(px * line_gap * 1.15)
    width = int(max(font.getlength(l) for l in lines)) + 8
    img = Image.new("RGBA", (max(width, 1), lh * len(lines) + int(px * 0.45)), (0, 0, 0, 0))
    d = ImageDraw.Draw(img)
    for i, l in enumerate(lines):
        d.text((2, i * lh), l, font=font, fill=color)
    return np.asarray(img)

def put_text_img(fig, text, x, y, px, color=B.INK, bold=False, ha="left", max_width=None, alpha=1.0, max_lines=None):
    """Place shaped text on a figure. x, y = figure fraction of the top-left anchor. Returns height in figure fraction."""
    arr = text_image(text, px, color, bold, max_width, max_lines=max_lines)
    if alpha < 1:
        arr = arr.copy(); arr[..., 3] = (arr[..., 3] * alpha).astype(np.uint8)
    W, H = fig.get_size_inches() * fig.dpi
    h, w = arr.shape[:2]
    xo = x * W - (w / 2 if ha == "center" else (w if ha == "right" else 0))
    fig.figimage(arr, xo=int(xo), yo=int(y * H - h), origin="upper", zorder=10)
    return h / H

_chk = plt.figure(figsize=(9, 1.6), dpi=100)
put_text_img(_chk, "ગુજરાતી તપાસ: સ્પષ્ટ ઉચ્ચારણ, ક્ષ ત્ર જ્ઞ ૧૨૩", 0.02, 0.92, 30, B.INK, bold=True)
put_text_img(_chk, "हिंदी जाँच: क्षत्रिय ज्ञान श्री", 0.02, 0.45, 30, B.INK, bold=True)
_chk.savefig("script_render_check.png", facecolor=B.PAPER); plt.close(_chk)
print("libraqm shaping:", HAS_RAQM, "→ saved script_render_check.png (open it to eyeball the conjuncts)")

## 2 · Data: the same sentences, read aloud in three languages (FLEURS test set)

FLEURS gives about 12 hours of read speech per language. Each FLEURS sentence is a translation of the same
FLoRes-101 sentence, so a sentence ID means the same content in Gujarati, Hindi and English. The notebook downloads
each language's test transcript (`test.tsv`) and test audio (`audio/test.tar.gz`, 16 kHz WAV) from the Hugging Face
dataset repo, keeps one recording per sentence, and picks up to `N_PER_LANG` sentence IDs that exist in all three
languages (seeded, so the selection is reproducible). It extracts only those clips from each archive.

In [ ]:
# CELL: data
def _hf_path(lang, kind):
    """Locate FLEURS files for `lang` in the HF repo. kind = 'tsv' or 'audio'. Tries the documented layout first."""
    from huggingface_hub import hf_hub_download, HfApi
    first = f"data/{lang}/test.tsv" if kind == "tsv" else f"data/{lang}/audio/test.tar.gz"
    try:
        return hf_hub_download(FLEURS_REPO, first, repo_type="dataset")
    except Exception as e:
        print(f"  {first} not found directly ({type(e).__name__}); searching the repo file list")
    files = HfApi().list_repo_files(FLEURS_REPO, repo_type="dataset")
    want = (lambda f: f.endswith("test.tsv")) if kind == "tsv" else (lambda f: f.endswith("test.tar.gz"))
    cands = [f for f in files if f"/{lang}/" in f and want(f) and "parquet" not in f]
    if not cands:
        raise FileNotFoundError(f"no {kind} file for {lang} in {FLEURS_REPO}")
    return hf_hub_download(FLEURS_REPO, cands[0], repo_type="dataset")

def read_tsv(path):
    """FLEURS TSV, no header: id, file name, raw transcription, normalised transcription, chars, num_samples, gender."""
    rows = []
    for line in open(path, encoding="utf-8"):
        p = line.rstrip("\n").split("\t")
        if len(p) < 7 or not p[5].strip().isdigit():
            continue
        rows.append({"sid": p[0].strip(), "file": p[1].strip(), "raw": p[2].strip(), "norm_ref": p[3].strip(),
                     "num_samples": int(p[5]), "gender": p[6].strip().upper()})
    return pd.DataFrame(rows)

def load_wavs(tar_path, wanted):
    """Extract only the wanted file names from a FLEURS audio archive. Returns {file name: (float32 mono, sr)}."""
    import soundfile as sf
    out = {}
    with tarfile.open(tar_path, "r:gz") as tar:
        for m in tar:
            name = os.path.basename(m.name)
            if m.isfile() and name in wanted:
                a, sr = sf.read(io.BytesIO(tar.extractfile(m).read()), dtype="float32")
                out[name] = (a.mean(axis=1) if a.ndim > 1 else a, sr)
                if len(out) == len(wanted):
                    break
    return out

def to_16k(a, sr):
    if sr == 16000:
        return a
    from scipy.signal import resample_poly
    g = math.gcd(int(sr), 16000)
    return resample_poly(a, 16000 // g, int(sr) // g).astype(np.float32)

def load_fleurs():
    rng = np.random.default_rng(SEED)
    tables = {}
    for lang in LANGS:
        t = read_tsv(_hf_path(lang, "tsv"))
        t = t.sample(frac=1.0, random_state=SEED).drop_duplicates("sid")    # one recording per sentence (random)
        tables[lang] = t.set_index("sid")
        print(f"{lang}: {len(t)} test sentences with audio")
    common = sorted(set.intersection(*[set(t.index) for t in tables.values()]))
    parallel = len(common) >= N_PER_LANG
    common_ids = list(rng.choice(common, N_PER_LANG, replace=False)) if parallel else None
    if not parallel:
        print(f"!! Only {len(common)} sentence IDs are shared by all three languages: sampling each language separately")
    clips = []
    for lang, t in tables.items():
        ids = common_ids if parallel else list(rng.choice(t.index.values, min(N_PER_LANG, len(t)), replace=False))
        sel = t.loc[ids]
        wavs = load_wavs(_hf_path(lang, "audio"), set(sel.file))
        for sid, r in sel.iterrows():
            if r.file not in wavs:
                continue
            a, sr = wavs[r.file]
            clips.append({"lang": lang, "sid": sid, "file": r.file, "ref_raw": r.raw, "gender": r.gender,
                          "sr_in": sr, "audio": to_16k(a, sr)})
    return pd.DataFrame(clips), parallel

def load_stand_in():
    """SMOKE_TEST only: no network. Synthetic noise bursts with short sentences we wrote ourselves (not FLEURS)."""
    sents = {
        "gu_in": ["આજે અમદાવાદમાં વરસાદ પડ્યો", "બજારમાં શાકભાજીના ભાવ વધ્યા છે", "ટ્રેન સવારે ૧૦ વાગ્યે આવશે",
                  "શાળામાં ૨૫ બાળકો છે", "ખેડૂતો પાણીની રાહ જુએ છે", "આ રસ્તો ગાંધીનગર તરફ જાય છે"],
        "hi_in": ["आज अहमदाबाद में बारिश हुई", "बाज़ार में सब्ज़ियों के दाम बढ़े हैं", "ट्रेन सुबह १० बजे आएगी",
                  "स्कूल में २५ बच्चे हैं", "किसान पानी का इंतज़ार कर रहे हैं", "यह सड़क गांधीनगर की ओर जाती है"],
        "en_us": ["It rained in Ahmedabad today", "Vegetable prices have gone up in the market",
                  "The train will arrive at 10 in the morning", "There are 25 children in the school",
                  "Farmers are waiting for water", "This road goes towards Gandhinagar"],
    }
    rng = np.random.default_rng(SEED); clips = []
    for lang, ss in sents.items():
        for i in range(12):
            s = ss[i % len(ss)]; n = int(16000 * (1.2 + 0.25 * len(s.split())))
            clips.append({"lang": lang, "sid": f"S{i:02d}", "file": f"synthetic_{lang}_{i:02d}.wav", "ref_raw": s,
                          "gender": "MALE" if i % 2 else "FEMALE", "sr_in": 16000,
                          "audio": (0.05 * rng.standard_normal(n)).astype(np.float32)})
    return pd.DataFrame(clips), True

t0 = time.time()
CLIPS, PARALLEL = load_stand_in() if SMOKE_TEST else load_fleurs()
CLIPS["dur_s"] = CLIPS.audio.map(len) / 16000
N_TRIMMED = int((CLIPS.dur_s > MAX_AUDIO_S).sum())
CLIPS["audio"] = CLIPS.audio.map(lambda a: a[: int(MAX_AUDIO_S * 16000)])
N_CLIPS = CLIPS.groupby("lang").size().reindex(list(LANGS)).fillna(0).astype(int)
DATA_TABLE = CLIPS.groupby("lang").agg(clips=("sid", "size"), minutes=("dur_s", lambda s: s.sum() / 60),
                                       median_s=("dur_s", "median"),
                                       female=("gender", lambda g: (g == "FEMALE").mean())).reindex(list(LANGS))
print(DATA_TABLE.round(2).to_string())
print(f"\nParallel sentences across languages: {PARALLEL} · trimmed to {MAX_AUDIO_S:.0f}s: {N_TRIMMED} · "
      f"loaded in {time.time() - t0:.0f}s")

fig, ax = plt.subplots(figsize=(9, 3.8))
for i, lang in enumerate(LANGS):
    ax.hist(CLIPS[CLIPS.lang == lang].dur_s, bins=20, alpha=0.75, color=B.SERIES_COLORS[i], label=LANGS[lang][0])
ax.set_xlabel("clip length (seconds)"); ax.set_ylabel("clips"); ax.legend()
B.chart_title(ax, "The test clips", f"{int(N_CLIPS.min())}–{int(N_CLIPS.max())} clips per language · FLEURS test split")
B.source_note(fig, "23N Field Test #09 · " + DATA_CREDIT + " · Independent study on public data.")
plt.tight_layout(); plt.show()

## 3 · Models

Each model is loaded, run on every language it supports, timed, and freed before the next one, so the whole run fits
a 15 GB T4. Whisper models get the language forced (`language="gu"`, `task="transcribe"`), so the test measures
transcription, not language identification. MMS swaps in its Gujarati, Hindi or English adapter. Decoding is greedy
for every model, so the run is reproducible on the same hardware and library versions.

In [ ]:
# CELL: models
def _free():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

def load_asr(key, hf_id, family):
    """Returns (transcribe(list of 16 kHz arrays, fleurs lang) -> list of str, params in billions)."""
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if dev == "cuda" else torch.float32
    if family == "whisper":
        from transformers import WhisperForConditionalGeneration, WhisperProcessor, GenerationConfig
        proc = WhisperProcessor.from_pretrained(hf_id)
        model = WhisperForConditionalGeneration.from_pretrained(hf_id, torch_dtype=dtype).to(dev).eval()
        if not getattr(model.generation_config, "lang_to_id", None):
            # older fine-tunes ship a generation config without language tokens; borrow the base model's
            model.generation_config = GenerationConfig.from_pretrained("openai/whisper-small")
        model.generation_config.forced_decoder_ids = None    # the language/task come from generate() below
        model.config.forced_decoder_ids = None
        def transcribe(arrs, lang):
            feats = proc.feature_extractor(arrs, sampling_rate=16000, return_tensors="pt").input_features
            with torch.no_grad():
                ids = model.generate(feats.to(dev, dtype), language=LANGS[lang][1], task="transcribe",
                                     max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False)
            return [s.strip() for s in proc.batch_decode(ids, skip_special_tokens=True)]
    else:
        from transformers import AutoProcessor, Wav2Vec2ForCTC
        proc = AutoProcessor.from_pretrained(hf_id)
        model = Wav2Vec2ForCTC.from_pretrained(hf_id).to(dev).eval()
        state = {"lang": None}
        def transcribe(arrs, lang):
            code = LANGS[lang][2]
            if state["lang"] != code:
                proc.tokenizer.set_target_lang(code); model.load_adapter(code); state["lang"] = code
            inp = proc(arrs, sampling_rate=16000, return_tensors="pt", padding=True)
            mask = inp.get("attention_mask", None)
            with torch.no_grad():
                logits = model(inp.input_values.to(dev), attention_mask=None if mask is None else mask.to(dev)).logits
            return [s.strip() for s in proc.batch_decode(logits.argmax(-1))]
    params = sum(p.numel() for p in model.parameters()) / 1e9
    return transcribe, params

def load_stand_in_asr(key, hf_id, family):
    """SMOKE_TEST only: returns the reference with seeded word drops and swaps, at a made-up rate per model and
    language, so every downstream cell has something to chew on. Says nothing about any real model."""
    rate = {"whisper-small": 0.5, "whisper-large-v3-turbo": 0.25, "mms-1b-all": 0.3, "whisper-gujarati-small": 0.15}[key]
    lang_mult = {"gu_in": 1.0, "hi_in": 0.6, "en_us": 0.25}
    def transcribe(arrs, lang, refs=None):
        out = []
        for a in arrs:
            ref = CLIPS_BY_ID.get(id(a), "")
            r = np.random.default_rng(zlib.crc32(f"{key}|{lang}|{len(a)}|{ref}".encode()))
            words = [w for w in ref.split() if r.random() > rate * lang_mult[lang] * 0.5]
            words = [w[::-1] if r.random() < rate * lang_mult[lang] * 0.5 else w for w in words]
            if key == "whisper-small" and lang == "gu_in" and r.random() < 0.2:
                words = words + words[-3:] * 4     # mimic a runaway repeat
            out.append(" ".join(words))
        return out
    return transcribe, 0.0

if SMOKE_TEST:
    CLIPS_BY_ID = {id(a): s for a, s in zip(CLIPS.audio, CLIPS.ref_raw)}
print("Models to run:", [m[0] for m in MODELS])

## 4 · Method: one normaliser for every language, then WER and CER

Both the FLEURS reference (`raw transcription`) and each model's output go through the same normaliser: Unicode NFC,
lower case, Gujarati and Devanagari digits mapped to 0–9, punctuation and symbols turned into spaces, whitespace
collapsed. Combining vowel signs and viramas are **kept**. (Whisper's `BasicTextNormalizer` replaces every Unicode
mark with a space, which splits Gujarati words apart and distorts WER for Indic scripts, so we don't use it.)

- **WER** = (substitutions + deletions + insertions) / reference words, pooled over all clips of a language.
- **CER** = the same on characters. It is kinder to Gujarati's long, inflected words, so we report both.
- **Confidence intervals**: 95% bootstrap over clips (`N_BOOT` resamples).
- A number written as a word in one text and as digits in the other counts as an error. That is deliberate: a voice
  bot that hears "દસ" and writes "10" is fine, but this scorer can't tell, so number errors are reported separately.

In [ ]:
# CELL: scoring
import jiwer
DIGITS = str.maketrans("૦૧૨૩૪૫૬૭૮૯०१२३४५६७८९", "01234567890123456789")

def normalise(s):
    s = unicodedata.normalize("NFC", str(s)).lower().translate(DIGITS)
    s = "".join(" " if unicodedata.category(c)[0] in "PSZC" and c != " " else c for c in s)
    return re.sub(r"\s+", " ", s).strip()

def utt_errors(ref, hyp):
    """Word and character error counts for one clip, plus the word alignment."""
    if not hyp:     # an empty output = every reference word (and character) deleted
        return dict(w_err=len(ref.split()), w_ref=len(ref.split()), c_err=len(ref), c_ref=len(ref), align=None)
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return dict(w_err=w.substitutions + w.deletions + w.insertions, w_ref=len(ref.split()),
                c_err=c.substitutions + c.deletions + c.insertions, c_ref=len(ref), align=w)

def ref_word_status(al):
    """For one alignment: list of (ref word, correct?) and list of (ref word, hyp word) substitutions."""
    ref, hyp = al.references[0], al.hypotheses[0]
    status, subs = [], []
    for ch in al.alignments[0]:
        span = ref[ch.ref_start_idx:ch.ref_end_idx]
        if ch.type == "equal":
            status += [(w, True) for w in span]
        elif ch.type in ("substitute", "delete"):
            status += [(w, False) for w in span]
            if ch.type == "substitute":
                subs += list(zip(span, hyp[ch.hyp_start_idx:ch.hyp_end_idx]))
    return status, subs

assert normalise("ગુજરાતી, ભાષા! ૧૨") == "ગુજરાતી ભાષા 12"
assert utt_errors("a b c", "a x c")["w_err"] == 1
print("normaliser + scorer ready · example:", normalise("સ્પષ્ટ ઉચ્ચારણ, ૨૦૨૪!"))

In [ ]:
# CELL: run_eval
ROWS, MODEL_INFO = [], {}
CLIPS["ref"] = CLIPS.ref_raw.map(normalise)
N_EMPTY_REF = int((CLIPS.ref == "").sum())
CLIPS = CLIPS[CLIPS.ref != ""].reset_index(drop=True)          # nothing to score against
for key, hf_id, family, licence, langs in MODELS:
    t_load = time.time()
    try:
        transcribe, params = (load_stand_in_asr if SMOKE_TEST else load_asr)(key, hf_id, family)
    except Exception as e:
        print(f"!! {key} failed to load: {type(e).__name__}: {str(e)[:200]}"); continue
    MODEL_INFO[key] = {"hf_id": hf_id, "licence": licence, "params_b": params, "load_s": time.time() - t_load}
    for lang in langs:
        sub = CLIPS[CLIPS.lang == lang].sort_values("dur_s")          # similar lengths per batch = less padding
        t = time.time(); hyps = []
        for i in range(0, len(sub), BATCH):
            hyps += transcribe(list(sub.audio.iloc[i:i + BATCH]), lang)
        wall = time.time() - t
        audio_s = float(sub.dur_s.clip(upper=MAX_AUDIO_S).sum())
        for (_, r), h in zip(sub.iterrows(), hyps):
            hn = normalise(h)
            e = utt_errors(r.ref, hn)
            ROWS.append({"model": key, "lang": lang, "sid": r.sid, "gender": r.gender, "dur_s": r.dur_s,
                         "ref": r.ref, "ref_raw": r.ref_raw, "hyp_raw": h, "hyp": hn, **{k: v for k, v in e.items() if k != "align"},
                         "alignment": e["align"], "hyp_words": len(hn.split()),
                         "script_share": script_share(h, LANGS[lang][3])})
        MODEL_INFO[key][f"rtf_{lang}"] = audio_s / max(wall, 1e-6)
        sel = [x for x in ROWS if x["model"] == key and x["lang"] == lang]
        wer = sum(x["w_err"] for x in sel) / max(1, sum(x["w_ref"] for x in sel))
        print(f"{key:24s} {lang}: WER {wer:6.1%} · {len(sel)} clips · {audio_s / max(wall, 1e-6):5.1f}× real time")
    del transcribe; _free()

res = pd.DataFrame(ROWS)
res.drop(columns=["alignment"]).to_csv("ft09_transcripts.csv", index=False)
print("saved ft09_transcripts.csv;", len(res), "rows")

## 5 · Results: the headline number, after the sanity checks

In [ ]:
# CELL: results
def _boot(err, ref, n=N_BOOT, seed=SEED):
    r = np.random.default_rng(seed); idx = r.integers(0, len(err), (n, len(err)))
    b = err[idx].sum(1) / np.maximum(ref[idx].sum(1), 1)
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

summ = []
for (m, lang), g in res.groupby(["model", "lang"]):
    lo, hi = _boot(g.w_err.values, g.w_ref.values)
    summ.append({"model": m, "lang": lang, "clips": len(g), "WER": g.w_err.sum() / g.w_ref.sum(), "WER_lo": lo,
                 "WER_hi": hi, "CER": g.c_err.sum() / g.c_ref.sum(),
                 "WER_mean_per_clip": float((g.w_err / g.w_ref).mean()),
                 "runaway": int(((g.hyp_words > RUNAWAY_RATIO * g.w_ref) & (g.hyp_words > g.w_ref + 10)).sum()),
                 "empty": int((g.hyp == "").sum()), "script_ok": float((g.script_share >= 0.5).mean()),
                 "x_realtime": MODEL_INFO[m].get(f"rtf_{lang}", float("nan"))})
SUMMARY = pd.DataFrame(summ)
WER = SUMMARY.pivot(index="model", columns="lang", values="WER").reindex([m[0] for m in MODELS if m[0] in MODEL_INFO])
WER = WER.reindex(columns=[l for l in LANGS if l in WER.columns])

def _best(lang, exclude=()):
    s = SUMMARY[(SUMMARY.lang == lang) & ~SUMMARY.model.isin(exclude)]
    return s.sort_values("WER").iloc[0] if len(s) else None
b_gu, b_gu_gen, b_en, b_hi = _best("gu_in"), _best("gu_in", (FINETUNED_GU,)), _best("en_us"), _best("hi_in")
BEST_GU, GU_WER, GU_LO, GU_HI = b_gu.model, float(b_gu.WER), float(b_gu.WER_lo), float(b_gu.WER_hi)
GU_CER = float(b_gu.CER)
BEST_GU_GEN = b_gu_gen.model if b_gu_gen is not None else BEST_GU
GU_WER_GEN = float(b_gu_gen.WER) if b_gu_gen is not None else GU_WER
BEST_EN, EN_WER = (b_en.model, float(b_en.WER)) if b_en is not None else ("n/a", float("nan"))
BEST_HI, HI_WER = (b_hi.model, float(b_hi.WER)) if b_hi is not None else ("n/a", float("nan"))
GAP = GU_WER / EN_WER if EN_WER and EN_WER > 0 else float("nan")
N_MODELS = len(MODEL_INFO)
N_UTT = int(N_CLIPS.get("gu_in", 0))

# ---------------- sanity checks (before any headline is printed) ----------------
PROBLEMS, NOTES = [], []
for lang, n in N_CLIPS.items():
    if n < N_MIN:
        PROBLEMS.append(f"only {n} usable {lang} clips (< N_MIN={N_MIN}); a download or extraction probably failed")
for lang, (label, _, _, script) in LANGS.items():
    sh = CLIPS[CLIPS.lang == lang].ref_raw.map(lambda s: script_share(s, script)).median()
    if sh < 0.8:
        PROBLEMS.append(f"{label} references are only {sh:.0%} {script} script: wrong FLEURS config or a broken TSV parse")
if (CLIPS.sr_in != 16000).any():
    NOTES.append(f"{int((CLIPS.sr_in != 16000).sum())} clips were not 16 kHz and were resampled")
if not CLIPS.dur_s.between(0.5, 40).mean() >= 0.95:
    PROBLEMS.append("more than 5% of clips are shorter than 0.5 s or longer than 40 s: audio was decoded wrongly")
if len(MODEL_INFO) < len(MODELS):
    PROBLEMS.append(f"{len(MODELS) - len(MODEL_INFO)} model(s) failed to load: {[m[0] for m in MODELS if m[0] not in MODEL_INFO]}")
if not SUMMARY.WER.between(0, 5).all():
    PROBLEMS.append("a WER is negative, NaN or above 500%: the scorer is broken")
if "whisper-large-v3-turbo" in WER.index and "en_us" in WER.columns and WER.loc["whisper-large-v3-turbo", "en_us"] > EN_WER_CEILING:
    PROBLEMS.append(f"whisper-large-v3-turbo English WER is {WER.loc['whisper-large-v3-turbo', 'en_us']:.0%} "
                    f"(> {EN_WER_CEILING:.0%}) on read speech: the audio, language code or normaliser is broken")
if GU_WER < EN_WER:
    PROBLEMS.append("Gujarati beat English on the same sentences, which contradicts every published FLEURS result: "
                    "check the language codes and the TSV columns")
if FINETUNED_GU in WER.index and WER.loc[FINETUNED_GU, "gu_in"] > FT_WER_TOLERANCE * REPORTED_FT_WER:
    PROBLEMS.append(f"{FINETUNED_GU} WER {WER.loc[FINETUNED_GU, 'gu_in']:.0%} is over {FT_WER_TOLERANCE:g}× the "
                    f"{REPORTED_FT_WER:.0%} its authors report for the base variant: our decoding or scoring disagrees with theirs")
if not PARALLEL:
    NOTES.append("languages were sampled separately (not the same sentences): the English-Gujarati gap mixes content and language")
if N_TRIMMED:
    NOTES.append(f"{N_TRIMMED} clips were trimmed to {MAX_AUDIO_S:.0f} s")
if not HAS_GPU and not SMOKE_TEST:
    PROBLEMS.append("CPU run: reduced model set and clip count; run on a T4 for the headline")
if SMOKE_TEST:
    PROBLEMS.append("SMOKE_TEST is on: synthetic clips and stand-in models")
for n_ in NOTES:
    print("Note:", n_)
if PROBLEMS:
    print("!" * 100)
    for p in PROBLEMS: print("!! SANITY CHECK FAILED:", p)
    print("!! Do NOT post these numbers until every check above passes.")
    print("!" * 100)
else:
    print("All sanity checks passed.")

def _pct(x): return f"{100 * x:.0f}%"
HEADLINE = (f"The best small open model we tested ({BEST_GU}) got {_pct(GU_WER)} of spoken Gujarati words wrong, "
            f"against {_pct(EN_WER)} in English for the same sentences ({BEST_EN}).")
print()
print("HEADLINE (estimate, FLEURS test clips):", HEADLINE)
print(f"  Gujarati WER {GU_WER:.1%} (95% CI {GU_LO:.1%}–{GU_HI:.1%}), CER {GU_CER:.1%}, {N_UTT} clips")
print(f"  Best general multilingual model on Gujarati: {BEST_GU_GEN} at {GU_WER_GEN:.1%} WER"
      + (f" ({FINETUNED_GU} is fine-tuned on FLEURS train+dev: best case)" if BEST_GU == FINETUNED_GU else ""))
print(f"  Hindi best: {BEST_HI} {HI_WER:.1%} · English best: {BEST_EN} {EN_WER:.1%} · Gujarati/English gap {GAP:.1f}×")
print("  Robustness (mean of per-clip WER instead of pooled):")
print(SUMMARY.pivot(index="model", columns="lang", values="WER_mean_per_clip").round(3).to_string())
g_split = res[res.lang == "gu_in"].groupby(["model", "gender"]).apply(lambda g: g.w_err.sum() / g.w_ref.sum()).unstack()
print("  Robustness (Gujarati WER by speaker gender):"); print(g_split.round(3).to_string())
print(f"  Assumptions: N_PER_LANG={N_PER_LANG}, SEED={SEED}, MAX_AUDIO_S={MAX_AUDIO_S}, greedy decoding, "
      f"MAX_NEW_TOKENS={MAX_NEW_TOKENS}, RUNAWAY_RATIO={RUNAWAY_RATIO}, EN_WER_CEILING={EN_WER_CEILING}, N_BOOT={N_BOOT}")
print()
print(SUMMARY.round(3).to_string(index=False))
SUMMARY.to_csv("ft09_summary.csv", index=False)

In [ ]:
# CELL: results_chart
LANG_COLORS = {"en_us": B.MOSS, "hi_in": B.AMBER, "gu_in": B.INK}
def draw_results(ax, big=False, grow=1.0):
    models = list(WER.index); langs = [l for l in ("en_us", "hi_in", "gu_in") if l in WER.columns]
    x = np.arange(len(models)); w = 0.8 / len(langs)
    ymax = float(np.nanmax(WER.values)) * 1.25 if np.isfinite(np.nanmax(WER.values)) else 1
    for j, lang in enumerate(langs):
        vals = WER[lang].values
        bars = ax.bar(x + (j - (len(langs) - 1) / 2) * w, np.nan_to_num(vals) * 100 * grow, w * 0.92,
                      color=LANG_COLORS[lang], label=LANGS[lang][0])
        for b, v in zip(bars, vals):
            if np.isfinite(v) and grow > 0.95:
                ax.text(b.get_x() + b.get_width() / 2, v * 100 + ymax * 1.5, f"{100 * v:.0f}", ha="center",
                        va="bottom", fontsize=15 if big else 9, color=B.INK, fontproperties=B.FP.get("sans_bold"))
    ax.set_xticks(x, [m.replace("whisper-", "whisper-\n") for m in models], fontsize=14 if big else 9)
    ax.set_ylim(0, ymax * 100); ax.set_ylabel("word error rate, % (lower is better)", fontsize=13 if big else 10)
    ax.tick_params(axis="y", labelsize=12 if big else 9)
    ax.legend(loc="upper right", fontsize=14 if big else 9, ncol=len(langs))

fig, ax = plt.subplots(figsize=(9.5, 4.8))
draw_results(ax)
B.chart_title(ax, "Same sentences, three languages", f"{N_MODELS} open models · FLEURS test · {N_UTT} clips per language")
B.source_note(fig, "23N Field Test #09 · " + DATA_CREDIT + f" · {FINETUNED_GU} is fine-tuned on FLEURS train+dev · Independent study on public data.")
plt.tight_layout(); fig.savefig("ft09_results.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.2))
gu = SUMMARY[SUMMARY.lang == "gu_in"]
ax.scatter(gu.x_realtime, gu.WER * 100, s=120, color=B.MOSS, zorder=3)
for _, r in gu.iterrows():
    ax.annotate(r.model, (r.x_realtime, r.WER * 100), xytext=(8, 4), textcoords="offset points", fontsize=10, color=B.INK)
ax.set_xscale("log"); ax.set_xlabel("speed: seconds of audio per second of compute (log)"); ax.set_ylabel("Gujarati WER, %")
B.chart_title(ax, "Speed against accuracy, Gujarati", f"greedy decoding · batch {BATCH} · {HARDWARE}")
B.source_note(fig, "23N Field Test #09 · " + DATA_CREDIT)
plt.tight_layout(); fig.savefig("ft09_speed.png", dpi=150); plt.show()

## 6 · Where the models fail on Gujarati

Every reference word is marked right or wrong from the alignment, then grouped: **numbers** (any digit after
normalisation), **Latin-script words** (names and English terms written in Latin letters in the reference), and
**other words**. The notebook also counts runaway outputs (the model repeats itself past the end of the speech),
error rate by sentence length, and the word swaps the best model makes most often.

In [ ]:
# CELL: failure_analysis
def word_kind(w):
    return "numbers" if re.search(r"\d", w) else ("Latin-script words" if re.search(r"[a-z]", w) else "other words")

fail, SUBS = [], {}
for m, g in res[res.lang == "gu_in"].groupby("model"):
    cnt = collections.Counter(); bad = collections.Counter(); subs = collections.Counter()
    for _, r in g.iterrows():
        if r["alignment"] is None:
            st, sb = [(w, False) for w in r["ref"].split()], []
        else:
            st, sb = ref_word_status(r["alignment"])
        for w, ok in st:
            k = word_kind(w); cnt[k] += 1; bad[k] += (not ok)
        subs.update(sb)
    for k in ("numbers", "Latin-script words", "other words"):
        fail.append({"model": m, "kind": k, "words": cnt[k], "error_rate": bad[k] / cnt[k] if cnt[k] else float("nan")})
    SUBS[m] = subs
FAIL = pd.DataFrame(fail)
print("Gujarati: share of reference words the model got wrong, by kind of word")
print(FAIL.pivot(index="model", columns="kind", values="error_rate").round(3).to_string())
print(FAIL.groupby("kind").words.first().to_string())

gu_res = res[res.lang == "gu_in"].copy()
gu_res["len_bin"] = pd.cut(gu_res.w_ref, [0, 10, 20, 1000], labels=["≤10 words", "11–20 words", ">20 words"])
LEN_WER = gu_res.groupby(["model", "len_bin"], observed=False).apply(lambda g: g.w_err.sum() / g.w_ref.sum() if g.w_ref.sum() else float("nan")).unstack()
print("\nGujarati WER by sentence length:"); print(LEN_WER.round(3).to_string())

NUM_ERR = float(FAIL[(FAIL.model == BEST_GU) & (FAIL.kind == "numbers")].error_rate.iloc[0])
OTHER_ERR = float(FAIL[(FAIL.model == BEST_GU) & (FAIL.kind == "other words")].error_rate.iloc[0])
RUNAWAY_GU = SUMMARY[SUMMARY.lang == "gu_in"].set_index("model").runaway
TOP_SUBS = SUBS[BEST_GU].most_common(10)
print(f"\nBest Gujarati model ({BEST_GU}): numbers wrong {NUM_ERR:.0%} of the time, other words {OTHER_ERR:.0%}")
print("Runaway outputs on Gujarati:", RUNAWAY_GU.to_dict())
print("Most common word swaps (reference → output):")
for (a, b), n in TOP_SUBS:
    print(f"  {n:3d} × {a} → {b}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
piv = FAIL.pivot(index="model", columns="kind", values="error_rate").reindex(WER.index)[["numbers", "Latin-script words", "other words"]]
x = np.arange(len(piv)); w = 0.26
for j, k in enumerate(piv.columns):
    axes[0].bar(x + (j - 1) * w, np.nan_to_num(piv[k].values) * 100, w * 0.92, color=B.SERIES_COLORS[j], label=k)
axes[0].set_xticks(x, [m.replace("whisper-", "whisper-\n") for m in piv.index], fontsize=8)
axes[0].set_ylabel("reference words wrong, %"); axes[0].legend(fontsize=9)
B.chart_title(axes[0], "Which words go wrong", "Gujarati · share of reference words missed or swapped")
for j, m in enumerate(LEN_WER.index):
    axes[1].plot(range(LEN_WER.shape[1]), LEN_WER.loc[m].values * 100, "-o", color=B.SERIES_COLORS[j % 5], label=m)
axes[1].set_xticks(range(LEN_WER.shape[1]), list(LEN_WER.columns)); axes[1].set_ylabel("WER, %"); axes[1].legend(fontsize=8)
B.chart_title(axes[1], "Longer sentences", "Gujarati WER by reference length")
B.source_note(fig, "23N Field Test #09 · " + DATA_CREDIT + " · Independent study on public data.")
plt.tight_layout(); fig.savefig("ft09_failures.png", dpi=150); plt.show()

## 7 · Interactive demo
Record or upload a short clip in Gujarati (or Hindi or English) and see what the best Gujarati model hears. The cell
doesn't block "Run all"; open the public `gradio.live` link it prints. Audio you record goes to this Colab session
only.

In [ ]:
# CELL: gradio
try:
    if SMOKE_TEST:
        raise RuntimeError("skipped in smoke test")
    import gradio as gr
    _m = [m for m in MODELS if m[0] == BEST_GU][0]
    DEMO_ASR, _ = load_asr(_m[0], _m[1], _m[2])
    def gr_transcribe(path, lang_label):
        if not path:
            return "Record or upload a clip first."
        import soundfile as sf
        a, sr = sf.read(path, dtype="float32")
        a = to_16k(a.mean(axis=1) if a.ndim > 1 else a, sr)[: int(MAX_AUDIO_S * 16000)]
        lang = {v[0]: k for k, v in LANGS.items()}[lang_label]
        if lang not in _m[4]:
            return f"{BEST_GU} is evaluated on Gujarati only."
        t = time.time(); txt = DEMO_ASR([a], lang)[0]
        return f"**{txt}**\n\n<sub>{time.time() - t:.1f}s · {BEST_GU} · Independent study on public data by 23N AI Labs</sub>"
    with gr.Blocks(title="23N Field Test #09 · Spoken Gujarati") as demo:
        gr.Markdown(f"## What does a small open model hear?\n{BEST_GU} had a {_pct(GU_WER)} word error rate on FLEURS Gujarati test clips.")
        au = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Clip (up to 30 s)")
        lg = gr.Dropdown([v[0] for v in LANGS.values()], value="Gujarati", label="Language")
        out = gr.Markdown()
        gr.Button("Transcribe").click(gr_transcribe, [au, lg], out)
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo not started:", e)

## 8 · Video (1080×1080, 30 fps, 58 s timeline saved at 1.5×, H.264) and cover (1080×1350)
Every number and transcript in the video comes from the run above. The example clip is the first Gujarati clip
(in sentence-ID order) of 6–16 words where the best model's WER is between 15% and 60%, so it shows a typical
mistake rather than the best or worst case.

In [ ]:
# CELL: video_lib
VIDEO_PATH, COVER_PATH = f"ft{FT_NUM}_video.mp4", f"ft{FT_NUM}_cover.png"
FPS = 30
def _ease(p): p = min(max(p, 0.0), 1.0); return 1 - (1 - p) ** 3
def _prog(t, a, b): return min(max((t - a) / (b - a), 0.0), 1.0)

def pick_example():
    g = res[(res.lang == "gu_in") & (res.model == BEST_GU)].copy()
    g["wer"] = g.w_err / g.w_ref
    c = g[g.w_ref.between(6, 16) & g.wer.between(0.15, 0.6)].sort_values("sid")
    row = c.iloc[0] if len(c) else g.iloc[(g.wer - g.wer.median()).abs().argsort().iloc[0]]
    hyps = res[(res.lang == "gu_in") & (res.sid == row.sid)].set_index("model")
    return row, [(m, hyps.loc[m]) for m in WER.index if m in hyps.index]

def _short(s, n=110):
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + " …"

def draw_evidence(fig, t, ctx):
    """t runs 0..35 inside the evidence section (video seconds 5..40)."""
    B.header(fig, FT_NUM)
    if t < 7:                                                         # the test set
        fig.text(0.055, 0.85, "The test", fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
        fig.text(0.055, 0.785, ("SAME SENTENCES, READ ALOUD IN 3 LANGUAGES" if PARALLEL else "FLEURS TEST CLIPS, 3 LANGUAGES"),
                 fontproperties=B.FP.get("mono"), fontsize=15, color=B.MUTED, va="top")
        for i, lang in enumerate(ctx["langs"]):
            a = _prog(t, 0.4 + i * 0.8, 1.2 + i * 0.8)
            y = 0.66 - i * 0.16
            fig.text(0.055, y, LANGS[lang][0], fontproperties=B.FP.get("display"), fontsize=46, color=LANG_COLORS[lang], alpha=a, va="top")
            fig.text(0.52, y, f"{int(DATA_TABLE.loc[lang, 'clips'])} clips", fontproperties=B.FP.get("display"), fontsize=40,
                     color=B.INK, alpha=a, va="top")
            fig.text(0.52, y - 0.06, f"{DATA_TABLE.loc[lang, 'minutes']:.0f} MIN OF AUDIO", fontproperties=B.FP.get("mono"),
                     fontsize=14, color=B.MUTED, alpha=a, va="top")
        B.caption(fig, f"Google FLEURS test set · {ctx['n_models']} open models · forced language, greedy decoding")
    elif t < 20:                                                      # one clip, every model
        row, hyps = ctx["example"]
        fig.text(0.055, 0.85, "WHAT WAS SAID", fontproperties=B.FP.get("mono_bold"), fontsize=14, color=B.MUTED, va="top")
        put_text_img(fig, _short(row.ref_raw, 120), 0.055, 0.82, 36, B.INK, bold=True,
                     max_width=970, max_lines=2)
        for i, (m, h) in enumerate(hyps):
            t0 = 8.5 + i * 2.2
            if t < t0:
                continue
            a = _prog(t, t0, t0 + 0.6); y = 0.64 - i * 0.125
            wer = h.w_err / h.w_ref
            fig.text(0.055, y, f"{m.upper()} · WER {100 * wer:.0f}%", fontproperties=B.FP.get("mono_bold"), fontsize=13,
                     color=B.MOSS if m == BEST_GU else B.MUTED, alpha=a, va="top")
            put_text_img(fig, _short(h.hyp_raw, 100) or "(no output)", 0.055, y - 0.025, 26,
                         B.MOSS if m == BEST_GU else B.BODY, max_width=970, alpha=a, max_lines=2)
        B.caption(fig, "One Gujarati clip from the test set and what each model wrote")
    elif t < 29:                                                      # the scoreboard
        fig.text(0.055, 0.85, "Word error rate", fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
        fig.text(0.055, 0.785, f"% OF WORDS WRONG · LOWER IS BETTER · {N_UTT} CLIPS PER LANGUAGE",
                 fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")
        ax = fig.add_axes([0.10, 0.20, 0.85, 0.52]); draw_results(ax, big=True, grow=_ease(_prog(t, 20.3, 23)))
        B.caption(fig, f"Same sentences. Every model does worse in Gujarati than in English")
    else:                                                             # where it fails
        fig.text(0.055, 0.85, "Where it fails", fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
        fig.text(0.055, 0.785, f"{BEST_GU.upper()} · GUJARATI", fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")
        ax = fig.add_axes([0.37, 0.30, 0.55, 0.42])
        kinds = ["numbers", "Latin-script words", "other words"]
        vals = [ctx["fail"].get(k, float("nan")) for k in kinds]; gg = _ease(_prog(t, 29.2, 31.5))
        ax.barh(range(3)[::-1], [np.nan_to_num(v) * 100 * gg for v in vals], color=[B.MOSS, B.AMBER, B.STONE], height=0.6)
        ax.set_yticks(range(3)[::-1], kinds, fontsize=16); ax.set_xlim(0, 100); ax.tick_params(axis="x", labelsize=12)
        ax.set_xlabel("% of these words wrong", fontsize=13); ax.grid(axis="y", visible=False)
        for i, v in enumerate(vals):
            if np.isfinite(v) and gg > 0.95:
                ax.text(v * 100 + 2, 2 - i, f"{100 * v:.0f}%", va="center", fontsize=16, color=B.INK, fontproperties=B.FP.get("sans_bold"))
            elif not np.isfinite(v):
                ax.text(2, 2 - i, "none in these clips", va="center", fontsize=13, color=B.STONE)
        B.caption(fig, f"Runaway outputs on Gujarati: {ctx['runaway']}")

def render_video(ctx, path=VIDEO_PATH):
    v = B.Video(path, fps=FPS)
    B.hook_card(v.fig, FT_NUM, "Can a small open model understand spoken Gujarati?",
                f"{ctx['n_models']} open models · FLEURS test · same sentences in Gujarati, Hindi, English"); v.hold(5)
    n = int(35 * FPS); key0, buf = None, None
    for f in range(n):
        t = f / FPS
        key = (("d", min(round(t * 10) / 10, 3.0)) if t < 7 else ("e", round(t * 10) / 10) if t < 20 else
               ("s", min(round(t * 15) / 15, 23.1)) if t < 29 else ("f", min(round(t * 15) / 15, 31.6)))
        if key != key0:
            v.clear(); draw_evidence(v.fig, t, ctx); buf = v.grab(); key0 = key
        v.proc.stdin.write(buf); v.frames += 1
    v.clear(); B.headline_card(v.fig, FT_NUM, _pct(GU_WER),
                               f"of Gujarati words wrong for the best small open model, against {_pct(EN_WER)} in English",
                               f"{BEST_GU} · FLEURS test · {N_UTT} clips per language · word error rate"); v.hold(12)
    v.clear(); B.end_card(v.fig, FT_NUM, DATA_CREDIT); v.hold(6)
    v.close()
    return path

def render_cover(ctx, path=COVER_PATH):
    B.cover(path, FT_NUM, "Can a small open model understand spoken Gujarati?",
            f"{ctx['n_models']} models · same sentences · Gujarati, Hindi, English",
            _pct(GU_WER), f"of Gujarati words wrong for the best model. English: {_pct(EN_WER)}",
            image=lambda ax: draw_results(ax.inset_axes([0.09, 0.1, 0.91, 0.9]), big=True), data_credit=DATA_CREDIT)
    return path

def build_video_ctx():
    fb = FAIL[FAIL.model == BEST_GU].set_index("kind").error_rate.to_dict()
    run = ", ".join(f"{m} {int(n)}" for m, n in RUNAWAY_GU.items())
    return {"langs": [l for l in ("gu_in", "hi_in", "en_us") if l in LANGS], "n_models": N_MODELS,
            "example": pick_example(), "fail": fb, "runaway": run}

In [ ]:
# CELL: video_render
VCTX = build_video_ctx()
render_video(VCTX)
render_cover(VCTX)
_probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                         "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO_PATH],
                        capture_output=True, text=True)
print(_probe.stdout)
try:
    from google.colab import files
    files.download(VIDEO_PATH); files.download(COVER_PATH)
except Exception:
    print("Not on Colab: video and cover saved in the working directory.")

## 9 · Post text (ready to paste once the sanity checks pass)

In [ ]:
# CELL: post_text
POST_LINKEDIN_TEMPLATE = """Can a small open speech model understand spoken Gujarati?

23N Field Test #09: {headline}

Census 2011 counted about 5.5 crore people whose mother tongue is Gujarati. Phone lines, shops and helpdesks that want to take voice need a model that hears them.

How we tested it:
• {n_utt} clips per language from Google's FLEURS test set: the same sentences read in Gujarati, Hindi and English
• {n_models} open models on a free Colab GPU: {model_list}
• Word error rate, Gujarati: {gu_wer} for {best_gu} (95% CI {gu_lo}–{gu_hi}); best general multilingual model {gu_wer_gen} ({best_gu_gen}). Hindi best {hi_wer}, English best {en_wer}
• Where it fails: numbers wrong {num_err} of the time against {other_err} for other words

Limits: read speech by adult volunteers, not phone calls or dialects; {finetuned} was fine-tuned on FLEURS train data, so it is a best case. Some model licences are non-commercial.

Notebook in the comments.

{signature}
"""
POST_X_TEMPLATE = """Tested {n_models} small open speech models on the same sentences read in Gujarati, Hindi and English (FLEURS). Best Gujarati word error rate: {gu_wer} ({best_gu}). English: {en_wer}. 23N Field Test #09. Notebook below.

---
Thread (optional):
1/ {n_utt} test clips per language, forced language, greedy decoding, one script-safe normaliser. Best general multilingual model on Gujarati: {gu_wer_gen} ({best_gu_gen}).
2/ Where it fails: numbers {num_err} wrong vs {other_err} for other words. Runaway outputs: {runaway}.
3/ Limits: read speech, not calls or dialects; one model fine-tuned on FLEURS train. Independent study on public data.
"""
def _load_template(name, fallback):
    try:
        return open(name, encoding="utf-8").read()
    except Exception:
        return fallback
POST_VALUES = dict(headline=HEADLINE, n_utt=N_UTT, n_models=N_MODELS, model_list=", ".join(MODEL_INFO),
                   gu_wer=_pct(GU_WER), gu_lo=_pct(GU_LO), gu_hi=_pct(GU_HI), best_gu=BEST_GU,
                   gu_wer_gen=_pct(GU_WER_GEN), best_gu_gen=BEST_GU_GEN, hi_wer=_pct(HI_WER), en_wer=_pct(EN_WER),
                   best_en=BEST_EN, num_err=_pct(NUM_ERR), other_err=_pct(OTHER_ERR), finetuned=FINETUNED_GU,
                   runaway=VCTX["runaway"], gap=f"{GAP:.1f}×",
                   signature=B.post_signature(FT_NUM) + " #Gujarati #SpeechRecognition #OpenSourceAI")
post_li = _load_template("post_linkedin.md", POST_LINKEDIN_TEMPLATE).format(**POST_VALUES)
post_x = _load_template("post_x.md", POST_X_TEMPLATE).format(**POST_VALUES)
x_main = post_x.split("---")[0].strip()
if PROBLEMS:
    print("!! SANITY CHECKS FAILED (see Results). Do NOT post this text.\n")
print("=" * 30, f"LinkedIn ({len(post_li)} chars)", "=" * 30); print(post_li)
print("=" * 30, f"X (main post {len(x_main)} chars)", "=" * 30); print(post_x)
if len(post_li) > 1300: print(f"WARNING: LinkedIn post is {len(post_li)} chars (> 1,300): trim before posting")
if len(x_main) > 280: print(f"WARNING: X post is {len(x_main)} chars (> 280): trim before posting")

## 10 · Limitations (read before quoting the number)

- **Read speech, not real calls.** FLEURS is adult volunteers reading Wikipedia-style sentences into a microphone.
  Phone audio (8 kHz), background noise, spontaneous speech, code-mixing ("Gujlish") and regional dialects such as
  Kathiawadi or Surti will all be harder. Read the number as a ceiling for these models, not what a call centre gets.
- **A small sample.** Up to 150 clips per language. The bootstrap interval shows the sampling noise; differences of
  a few points between models are within it.
- **The fine-tuned model is a best case.** `vasista22/whisper-gujarati-small` was trained on FLEURS train + dev. The
  test clips are held out, but the recording style and domain match its training data.
- **Scoring choices matter.** WER depends on the normaliser. Numbers written as words in one text and digits in the
  other count as errors, and Gujarati's long inflected words make WER harsher than CER. Both are printed.
- **Forced language.** Whisper is told the language. In a product it would have to detect Gujarati first, which is
  a separate source of errors not measured here.
- **Licences.** FLEURS is CC BY 4.0. `facebook/mms-1b-all` is **CC-BY-NC 4.0 (non-commercial)**. The Whisper models
  are MIT and the Gujarati fine-tune Apache 2.0. Check each model card before any commercial use.